In [1]:
sql_server_name = "ops-int-wus-sqlserver-1-primary.database.windows.net"
db_port = 1433
db_name = "GoldDataMart"
linked_service_name = "GoldDataMart"
env = "int"

pipeline_name = "test_DimKytheraDetail"
pipeline_id = "-1"
step_name = "DimKytheraDetail"
params_json = ""


In [2]:
mssparkutils.notebook.run("Administrative/Functions", 90, {"function":"Write_To_Log","env":env,"pipeline_name":pipeline_name,"pipeline_id":pipeline_id,"step_name":step_name + "_Start","in_param":params_json})

In [3]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import Window

access_token = TokenLibrary.getConnectionString(linked_service_name)

import pytz
pst = pytz.timezone('US/Pacific')
ProcessDate = datetime.now(pst)

In [4]:
# DimKytheraDetailDF = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
#     .option("url", "jdbc:sqlserver://%s:%d" % (sql_server_name, db_port)) \
#     .option("dbtable", "Kythera.DimCandidateDetail") \
#     .option("accessToken", access_token) \
#     .option("encrypt", "true") \
#     .option("databaseName", db_name) \
#     .option("hostNameInCertificate", "*.database.windows.net") \
#     .load()


In [5]:
taxonomy_list = ['Organ Procurement Organization',
            'Taxi',
            'Peer Specialist',
            'Home Health Aide',
            'Public Health or Welfare',
            'Ambulance Land Transport',
            'Technician/Technologist Optician',
            'Technician/Technologist Contact Lens',
            'Contractor Vehicle Modifications',
            'Blood Bank',
            'Secured Medical Transport (VAN)',
            'Respite Care Respite Care Camp',
            'Clinical Ethicist',
            'Technician/Technologist Ophthalmic',
            'Technician/Technologist Ophthalmic Assistant',
            'Mastectomy Fitter',
            'Durable Medical Equipment & Medical Supplies Customized Equipment',
            'Technician',
            'Ambulance',
            'Technician/Technologist Orthoptist',
            'Clinical Medical Laboratory',
            'Durable Medical Equipment & Medical Supplies Oxygen Equipment & Supplies',
            'Occupational Therapy Assistant Driving and Community Mobility',
            'Local Education Agency (LEA)',
            'Program of All-Inclusive Care for the Elderly (PACE) Provider Organization',
            'Technician/Technologist Optometric Assistant',
            'Non-emergency Medical Transport (VAN)',
            'Physiological Laboratory',
            'Eyewear Supplier',
            'Dental Laboratory',
            'Technician/Technologist Contact Lens Fitter',
            'In Home Supportive Care',
            'Technician/Technologist Optometric Technician',
            'Emergency Response System Companies',
            'Private Vehicle',
            'Occupational Therapy Assistant Low Vision',
            'Case Management',
            'Music Therapist',
            'Religious Nonmedical Nursing Personnel',
            'Physical Therapy Assistant',
            'Voluntary or Charitable',
            'Day Training, Developmentally Disabled Services',
            'Hearing Aid Equipment',
            'Assistant Behavior Analyst',
            'Recreation Therapist',
            'Dance Therapist',
            'Driver',
            'Audiologist Assistive Technology Supplier',
            'Ambulance Water Transport',
            'Hospice Care, Community Based',
            'Early Intervention Provider Agency',
            'Durable Medical Equipment & Medical Supplies',
            'Supports Brokerage',
            'Home Delivered Meals',
            'Developmental Therapist',
            'Transportation Broker',
            'Technician Personal Care Attendant',
            'Religious Nonmedical Practitioner',
            'Home Infusion',
            'Technician/Technologist Ocularist',
            'Prosthetist',
            'Contractor',
            'Ambulance Air Transport',
            'Non-Pharmacy Dispensing Site',
            'Technician Attendant Care Provider',
            'Recreational Therapist Assistant',
            'Meals',
            'Transportation Network Company',
            'Assistant, Podiatric',
            'Nursing Care',
            'Interpreter',
            'Contractor Home Modifications',
            'Community/Behavioral Health',
            'Home Health',
            'Lactation Consultant, Non-RN',
            'Specialist Research Study',
            'Technician/Technologist',
            'Durable Medical Equipment & Medical Supplies Nursing Facility Supplies',
            'Pedorthist',
            'Drama Therapist',
            'Lodging',
            'Medical Foods Supplier',
            'Orthotist',
            'Adult Companion',
            'Eye Bank',
            'Homemaker',
            'Orthotic Fitter',
            'Massage Therapist',
            'Foster Care Agency',
            'Durable Medical Equipment & Medical Supplies Parenteral & Enteral Nutrition',
            'Behavior Technician',
            'Occupational Therapy Assistant',
            'Bus',
            'Specialist Research Data Abstracter/Coder',
            'Military Clinical Medical Laboratory',
            'Durable Medical Equipment & Medical Supplies Dialysis Equipment & Supplies',
            'Chore Provider',
            'Prosthetic/Orthotic Supplier',
            'Portable X-ray and/or Other Portable Diagnostic Imaging Supplier',
            'Art Therapist',
            'Personal Emergency Response Attendant']

ProcedureCodeList = ['HIPPS',
    'ICD9',
    'ICD10PCS',
    'CPT',
    'HCPC',
    'ICD10'
]

In [6]:
KytheraPersonDF = spark.sql(f"""
    Select Rendering_Provider_NPI AS NPI
        , Max(K.Service_To_Date) AS LastBilledDate
    From WinnowSilver.Kythera AS K
    Where Rendering_Provider_NPI is not null
        and K.Service_To_Date is not null
    Group By Rendering_Provider_NPI
;""")

In [7]:
delta_lake_container = 'abfss://silver@opsintwuslake1.dfs.core.windows.net'

delta_table_name = 'kythera_mx_claim_line'
delta_table_folder = '/DataIntegrations/Winnow/'
delta_lake_file_path = f'{delta_lake_container}/{delta_table_folder}/{delta_table_name}'

KytheraDeltaDF = spark.read.format("delta").load(delta_lake_file_path) \
    .select(F.col('rendering_provider_npi').alias("ProviderNPI")
        , F.col('organization_npi').alias("OrganizationNPI")
        , F.col('service_from_date').alias("ServiceFromDate")
        , F.col('principal_procedure_codeset').alias("PrincipalProcedureCodeset")
        , F.col('principal_procedure_code').alias("PrincipalProcedureCode")
        , F.col('Billing_State').alias("BillingState")
        , F.col('claimaddress_state').alias("ClaimAddressState")
        , F.col('patient_state').alias("PatientState")
        ) \
    .filter( (F.col('rendering_provider_npi').isNotNull())
        & (F.col('organization_npitypecode') == 2) 
        & (F.col('rendering_provider_npitypecode') == 1)
        & (F.col('service_from_date').isNotNull())
        & (~F.col('organization_primarytaxonomydescription').isin(taxonomy_list)))

In [8]:
delta_table_name = 'NPIData'
delta_table_folder = '/NPI/NPIData/'
delta_lake_file_path = f'{delta_lake_container}/{delta_table_folder}/{delta_table_name}'

NPIDeltaDF = spark.read.format("delta").load(delta_lake_file_path) \
    .select(F.col('npi').alias("NPI")
        , F.col('provider_business_practice_location_address_state_name').alias("BillingState")
        , F.col('provider_organization_name_legal_business_name').alias("PrimaryHospitalName")
		, F.col('provider_first_line_business_practice_location_address').alias("PrimaryHospitalAddress")
		, F.col('provider_second_line_business_practice_location_address').alias("PrimaryHospitalAddress1")
		, F.col('provider_business_practice_location_address_city_name').alias("PrimaryHospitalCity")
		, F.substring(F.col('provider_business_practice_location_address_postal_code'), 0, 6).alias("PrimaryHospitalZipCode")
		, F.col('provider_business_practice_location_address_telephone_number').alias("PrimaryHospitalPhone")
        )

In [9]:
InOutStateDF = NPIDeltaDF.join(KytheraDeltaDF, [(KytheraDeltaDF.ProviderNPI == NPIDeltaDF.NPI)]) \
    .groupBy('ProviderNPI') \
    .agg(
        F.avg(F.when(NPIDeltaDF.BillingState == KytheraDeltaDF.BillingState, F.lit(1)).otherwise(F.lit(0))).cast("Integer").alias("BillingStateMatch")
        , F.avg(F.when(NPIDeltaDF.BillingState == KytheraDeltaDF.ClaimAddressState, F.lit(1)).otherwise(F.lit(0))).cast("Integer").alias("ClaimAddressStateMatch")
        , F.avg(F.when(NPIDeltaDF.BillingState == KytheraDeltaDF.PatientState, F.lit(1)).otherwise(F.lit(0))).cast("Integer").alias("PatientStateMatch")
    ) \
    .select(KytheraDeltaDF.ProviderNPI.alias("Npi")
        , ((F.col('BillingStateMatch') + F.col('ClaimAddressStateMatch') + F.col('PatientStateMatch')) / 3).alias("InStateChange")
        , (1 - ((F.col('BillingStateMatch') + F.col('ClaimAddressStateMatch') + F.col('PatientStateMatch')) / 3)).alias("OutStateChange")
        )


# Calculate the date 18 months ago
eighteen_months_ago = F.add_months(F.current_date(), -18)

ProcedureCodeKytheraDF = KytheraDeltaDF \
    .filter(F.col('principal_procedure_codeset').isin(ProcedureCodeList)
        & (F.col('service_from_date') >= eighteen_months_ago)
        ) \
    .withColumn('PrincipalProcedureCodeSetCode', F.concat(F.lit('"'), 'PrincipalProcedureCodeset', 'PrincipalProcedureCode', F.lit('"'))) \
    .groupBy('ProviderNPI', 'PrincipalProcedureCodeSetCode') \
    .agg(F.count('*').alias('ProcedureCount')) \
    .filter(F.col('ProcedureCount') >= 10) \
    .orderBy('ProviderNPI', 'PrincipalProcedureCodeSetCode') \
    .groupBy('ProviderNPI') \
    .agg(
        F.concat_ws(', ', F.collect_list('PrincipalProcedureCodeSetCode')).alias('MedicalCode')
    )



# Create the Hospital History Affiliation column
previousClaims = (Window.partitionBy('ProviderNPI', 'OrganizationNPI')
            .orderBy('ServiceFromDateFormatted')
            .rowsBetween(-6, -1))

orderNumber = (Window.partitionBy('ProviderNPI','ServiceFromDateFormatted')
            .orderBy(F.desc('Previous6MonthsClaimCount'), F.desc('ClaimCount'), 'OrganizationNPI'))

affiliationList = (Window.partitionBy('ProviderNPI').orderBy('ServiceFromDateFormatted'))

primaryOrder = (Window.partitionBy('ProviderNPI').orderBy(F.col("ServiceFromDateFormatted").desc(), F.col("Previous6MonthsClaimCount").desc()))

CachedKytheraDF = KytheraDeltaDF \
    .withColumn('ServiceFromDateFormatted', F.date_format("ServiceFromDate", "yyyyMM")) \
    .groupBy(["ProviderNPI", "ServiceFromDateFormatted", "OrganizationNPI"]).count() \
    .withColumnRenamed("count","ClaimCount") \
    .withColumn('Previous6MonthsClaimCount', F.sum('ClaimCount').over(previousClaims)) \
    .withColumn('OrderNumber', F.row_number().over(orderNumber)) \
    .filter(F.col('OrderNumber') == 1) \
    .withColumn('AffiliationItem', F.concat(F.lit('["'), F.col('OrganizationNPI'), F.lit('",'), F.col('ServiceFromDateFormatted'), F.lit(']'))) \
    .withColumn('PreviousOrganizationNPI', F.lag('OrganizationNPI').over(affiliationList)) \
    .filter(~F.col('OrganizationNPI').eqNullSafe(F.col('PreviousOrganizationNPI'))) \
    .cache()





calculateExpYearRangeItem = F.expr(
    """Concat('"',
    left(ServiceFromDateFormatted, 4), ' ', 
    CASE
        WHEN right(ServiceFromDateFormatted,2) == '01' THEN 'Jan'
        WHEN right(ServiceFromDateFormatted,2) == '02' THEN 'Feb'
        WHEN right(ServiceFromDateFormatted,2) == '03' THEN 'Mar'
        WHEN right(ServiceFromDateFormatted,2) == '04' THEN 'Apr'
        WHEN right(ServiceFromDateFormatted,2) == '05' THEN 'May'
        WHEN right(ServiceFromDateFormatted,2) == '06' THEN 'Jun'
        WHEN right(ServiceFromDateFormatted,2) == '07' THEN 'Jul'
        WHEN right(ServiceFromDateFormatted,2) == '08' THEN 'Aug'
        WHEN right(ServiceFromDateFormatted,2) == '09' THEN 'Sept'
        WHEN right(ServiceFromDateFormatted,2) == '10' THEN 'Oct'
        WHEN right(ServiceFromDateFormatted,2) == '11' THEN 'Nov'
        WHEN right(ServiceFromDateFormatted,2) == '12' THEN 'Dec'
        ELSE 'Unknown' END,
    '-',
    CASE WHEN NextServiceFromDateFormatted IS NULL THEN 'Present'
        ELSE Concat(left(NextServiceFromDateFormatted, 4), 
                    ' ',
                    CASE
                        WHEN right(NextServiceFromDateFormatted,2) == '01' THEN 'Jan'
                        WHEN right(NextServiceFromDateFormatted,2) == '02' THEN 'Feb'
                        WHEN right(NextServiceFromDateFormatted,2) == '03' THEN 'Mar'
                        WHEN right(NextServiceFromDateFormatted,2) == '04' THEN 'Apr'
                        WHEN right(NextServiceFromDateFormatted,2) == '05' THEN 'May'
                        WHEN right(NextServiceFromDateFormatted,2) == '06' THEN 'Jun'
                        WHEN right(NextServiceFromDateFormatted,2) == '07' THEN 'Jul'
                        WHEN right(NextServiceFromDateFormatted,2) == '08' THEN 'Aug'
                        WHEN right(NextServiceFromDateFormatted,2) == '09' THEN 'Sept'
                        WHEN right(NextServiceFromDateFormatted,2) == '10' THEN 'Oct'
                        WHEN right(NextServiceFromDateFormatted,2) == '11' THEN 'Nov'
                        WHEN right(NextServiceFromDateFormatted,2) == '12' THEN 'Dec'
                        ELSE 'Unknown' END)
        END,
    '"'
    )
    """)


# Format the date as 'yyyyMM'
CurrentDateString = datetime.now().strftime('%Y%m')

GroupedKytheraDF = CachedKytheraDF \
    .withColumn("NextServiceFromDateFormatted", F.lead(F.col("ServiceFromDateFormatted"), 1, CurrentDateString).over(affiliationList)) \
    .withColumn("NextServiceFromAsDate", F.to_date(F.col("NextServiceFromDateFormatted"), "yyyyMM")) \
    .withColumn("ServiceFromAsDate", F.to_date(F.col("ServiceFromDateFormatted"), "yyyyMM")) \
    .withColumn("ExpYearRangeItem", F.concat(F.lit('['), calculateExpYearRangeItem, F.lit(']'))) \
    .withColumn("MonthsBetweenServiceDates", F.months_between(
                                                            F.col("NextServiceFromAsDate")
                                                            , F.col("ServiceFromAsDate"))) \
    .withColumn("ExpYearCountItem", F.concat(
                                        F.lit('"')
                                        , F.floor(F.col("MonthsBetweenServiceDates") / 12)
                                        , F.lit(' years ')
                                        , (F.col("MonthsBetweenServiceDates") % 12).cast("Integer")
                                        , F.lit(' months"')
                                    )
                ) \
    .orderBy('ProviderNPI', 'ServiceFromDateFormatted') \
    .groupBy('ProviderNPI') \
    .agg(
        F.concat_ws(', ', F.collect_list('AffiliationItem')).alias('HospitalHistoryAffiliation'),
        F.last('OrganizationNPI').alias('PrimaryHospitalIDs'),
        F.concat_ws(', ', F.collect_list('ExpYearRangeItem')).alias('ExpYearRange'),
        F.concat_ws(', ', F.collect_list('ExpYearCountItem')).alias('ExpYearCount'),
        F.concat(
                    F.lit('"')
                    , F.floor(
                                F.months_between(
                                    F.to_date(F.lit(CurrentDateString), 'yyyyMM')
                                    , F.first('ServiceFromAsDate')
                                    ) / 12
                            )
                    , F.lit(' years ')
                    , (F.months_between(
                            F.to_date(F.lit(CurrentDateString), 'yyyyMM')
                            , F.first('ServiceFromAsDate')
                        ) % 12
                        ).cast("Integer")
                    , F.lit(' months"')
                ).alias("TotalExperience"),
        F.round(F.mean(F.year('ServiceFromAsDate')), 0).cast("Integer").alias("AverageStartYear"),
        F.round(F.mean(F.year('NextServiceFromAsDate')), 0).cast("Integer").alias("AverageEndYear")
        ) 

    # .join(DimKytheraDetailDF, [(DimKytheraDetailDF.ProviderNPI == GroupedKytheraDF.ProviderNPI)]  
    #                             , "left_anti") \

# Join together and return only the new records to be added
KytheraRecordsDF = GroupedKytheraDF \
    .join(ProcedureCodeKytheraDF, [GroupedKytheraDF.ProviderNPI == ProcedureCodeKytheraDF.ProviderNPI], "left") \
    .join(InOutStateDF, [GroupedKytheraDF.ProviderNPI == InOutStateDF.Npi], "left") \
    .join(NPIDeltaDF, [GroupedKytheraDF.PrimaryHospitalIDs == NPIDeltaDF.NPI]) \
    .select( GroupedKytheraDF.ProviderNPI
        , F.concat(F.lit('['), GroupedKytheraDF.HospitalHistoryAffiliation, F.lit(']')).alias("HospitalHistoryAffiliation")
        , GroupedKytheraDF.PrimaryHospitalIDs
        , F.substring(F.concat(F.lit('['), ProcedureCodeKytheraDF.MedicalCode, F.lit(']')),1,8000).alias("MedicalCode")
        , F.concat(F.lit('['), GroupedKytheraDF.ExpYearRange, F.lit(']')).alias("ExpYearRange")
        , F.concat(F.lit('['), GroupedKytheraDF.ExpYearCount, F.lit(']')).alias("ExpYearCount")
        , GroupedKytheraDF.TotalExperience
        , F.concat(F.lit('['), GroupedKytheraDF.AverageStartYear, F.lit(']')).alias("AverageStartYear")
        , F.concat(F.lit('['), GroupedKytheraDF.AverageEndYear, F.lit(']')).alias("AverageEndYear")
        , InOutStateDF.InStateChange
        , InOutStateDF.OutStateChange
        , NPIDeltaDF.PrimaryHospitalName
        , NPIDeltaDF.PrimaryHospitalAddress
        , NPIDeltaDF.PrimaryHospitalAddress1
        , NPIDeltaDF.PrimaryHospitalCity
        , NPIDeltaDF.BillingState.alias("PrimaryHospitalState")
        , NPIDeltaDF.PrimaryHospitalZipCode
        , NPIDeltaDF.PrimaryHospitalPhone
        , F.lit(ProcessDate).alias("CreatedDate")
        , F.lit(ProcessDate).alias("UpdatedDate")
    )

In [10]:
# DimKytheraDetailDF = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
#     .option("url", "jdbc:sqlserver://%s:%d" % (sql_server_name, db_port)) \
#     .option("dbtable", "Kythera.DimCandidateDetail") \
#     .option("accessToken", access_token) \
#     .option("encrypt", "true") \
#     .option("databaseName", db_name) \
#     .option("hostNameInCertificate", "*.database.windows.net") \
#     .load()

In [11]:
from sqlalchemy import create_engine, text, inspect

userName = "synapseNotebookUser"
password = "SQLr0cks!"

# Set up the connection string for SQLAlchemy (adjust for your environment)
engine = create_engine(f"""mssql+pyodbc://{userName}:{password}@{sql_server_name}/{db_name}?driver=ODBC+Driver+18+for+SQL+Server""")


In [12]:

# Insert the Definitive DataFrame into a temporary table
KytheraRecordsDF.withColumn("CreatedDate", F.lit(ProcessDate)).toPandas().to_sql("DimKytheraCandidateDetail_temp", con=engine, if_exists="replace", index=False)


In [13]:
update_query = f"""
Merge Into Kythera.DimCandidateDetail as Target
Using (
        Select ProviderNPI
            , HospitalHistoryAffiliation
            , PrimaryHospitalIDs
            , MedicalCode
            , ExpYearRange
            , ExpYearCount
            , TotalExperience
            , AverageStartYear
            , AverageEndYear
            , InStateChange
            , OutStateChange
            , CreatedDate
        From dbo.DimKytheraCandidateDetail_temp
    ) AS Source
    ON TARGET.ProviderNPI = Source.ProviderNPI
When Matched and (Coalesce(TARGET.HospitalHistoryAffiliation, '') <> Coalesce(Source.HospitalHistoryAffiliation, '')
                OR Coalesce(Target.PrimaryHospitalIDs, '') <> Coalesce(Source.PrimaryHospitalIDs, '')
                OR Coalesce(Cast(Target.MedicalCode as VarChar(max)), '') <> Coalesce(Source.MedicalCode, '')
                OR Coalesce(Target.ExpYearRange, '') <> Coalesce(Source.ExpYearRange, '')
                OR COALESCE(Target.ExpYearCount, '') <> COALESCE(Source.ExpYearCount, '')
                OR COALESCE(TARGET.TotalExperience, '') <> COALESCE(Source.TotalExperience, '')
                OR COALESCE(TARGET.AverageStartYear, '') <> COALESCE(Source.AverageStartYear, '')
                OR COALESCE(Target.AverageEndYear, '') <> COALESCE(Source.AverageEndYear, '')
                OR COALESCE(Target.InStateChange, '') <> COALESCE(Source.InStateChange, '')
                OR COALESCE(Target.OutStateChange, -1) <> COALESCE(Source.OutStateChange, -1)
                )
    Then UPDATE Set Target.HospitalHistoryAffiliation = Source.HospitalHistoryAffiliation
                    , TARGET.PrimaryHospitalIDs = Source.PrimaryHospitalIDs
                    , TARGET.MedicalCode = Source.MedicalCode
                    , TARGET.ExpYearRange = Source.ExpYearRange
                    , TARGET.ExpYearCount = Source.ExpYearCount
                    , Target.TotalExperience = Source.TotalExperience
                    , TARGET.AverageStartYear = Source.AverageStartYear
                    , TARGET.AverageEndYear = Source.AverageEndYear
                    , TARGET.InStateChange = Source.InStateChange
                    , TARGET.OutStateChange = Source.OutStateChange
                    , Target.UpdatedDate = Source.CreatedDate
When Not Matched by TARGET
    Then Insert (ProviderNPI, HospitalHistoryAffiliation, PrimaryHospitalIDs, MedicalCode
                , ExpYearRange, ExpYearCount, TotalExperience, AverageStartYear
                , AverageEndYear, InStateChange, OutStateChange, CreatedDate, UpdatedDate)
        Values (Source.ProviderNPI, Source.HospitalHistoryAffiliation, Source.PrimaryHospitalIDs, Source.MedicalCode
                , Source.ExpYearRange, Source.ExpYearCount, Source.TotalExperience, Source.AverageStartYear
                , Source.AverageEndYear, Source.InStateChange, Source.OutStateChange, Source.CreatedDate, Source.CreatedDate);
"""


# Execute the query using text() for raw SQL (recommended for SQLAlchemy 1.4+)
with engine.connect() as connection:
    transaction = connection.begin()  # Start transaction
    try:
        connection.execute(text(update_query))  # Wrap query in text()
        transaction.commit()  # Commit the changes
        print("Update successful!")
    except Exception as e:
        transaction.rollback()  # Rollback if there's an error
        print("Error:", e)

In [16]:
mssparkutils.notebook.run("Administrative/Functions", 90, {"function":"Write_To_Log","env":env,"pipeline_name":pipeline_name,"pipeline_id":pipeline_id,"step_name":step_name + "_End","in_param":params_json})